# Notebook 85: Daily Entries + Hourly Exits Strategy

**Date**: 2026-01-25  
**Strategy**: Buy The Dip (daily) + Hourly Exits  
**Period**: 2020-02-02 → Present (when all derivatives data available)

---

## 🎯 Objective

Test multi-timeframe strategy:
- **Entries**: Daily Buy The Dip signals (4/5 conditions)
- **Exits**: Hourly distribution signals (faster response)

## 📊 Hypothesis

Using hourly data for exits should:
1. Catch distribution earlier (LTH-SOPR spikes)
2. Exit before major dumps
3. Improve risk-adjusted returns
4. Reduce max drawdown

## 📈 Data Requirements

**Start Date**: 2020-02-02 (when funding_rate becomes available)

### Daily (Entries)
- STH-MVRV, STH-SOPR (BRK)
- Realized Profit/Loss (BRK)
- Funding Rate (Glassnode)
- Liquidations Long/Short (Glassnode)

### Hourly (Exits)
- LTH-SOPR (Bitcoin Lab)
- MVRV (Bitcoin Lab)
- Price (Bitcoin Lab)

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Paths
DATA_DIR = Path('../data')
BRK_DAILY = DATA_DIR / 'brk' / 'daily'
BL_HOURLY = DATA_DIR / 'bl' / 'hourly'
GN_DAILY = DATA_DIR / 'glassnode' / 'daily'
GN_HOURLY = DATA_DIR / 'glassnode' / 'hourly'

# Backtest period (aligned with derivatives data)
START_DATE = '2020-02-02'
END_DATE = '2026-01-25'

print("✅ Setup complete")
print(f"📅 Backtest period: {START_DATE} → {END_DATE}")

---

## 1. Load Daily Data (Entries)

Load all Buy The Dip metrics at daily resolution.

In [ ]:
# Load daily metrics for Buy The Dip entry signals
print("Loading daily metrics...")

# On-chain metrics (BRK)
mvrv_sth_daily = pd.read_parquet(BRK_DAILY / 'mvrv_sth.parquet')
sopr_sth_daily = pd.read_parquet(BRK_DAILY / 'sopr_sth.parquet')
realized_profit_daily = pd.read_parquet(BRK_DAILY / 'realized_profit.parquet')
realized_loss_daily = pd.read_parquet(BRK_DAILY / 'realized_loss.parquet')
price_daily = pd.read_parquet(BRK_DAILY / 'price.parquet')

# Derivatives (Glassnode)
funding_rate_daily = pd.read_parquet(GN_DAILY / 'funding_rate.parquet')
liq_long_daily = pd.read_parquet(GN_DAILY / 'liquidations_long.parquet')
liq_short_daily = pd.read_parquet(GN_DAILY / 'liquidations_short.parquet')

# Normalize to 'time' column
for df in [mvrv_sth_daily, sopr_sth_daily, realized_profit_daily, realized_loss_daily, 
           price_daily, funding_rate_daily, liq_long_daily, liq_short_daily]:
    if 'time' not in df.columns:
        df.reset_index(inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    df.sort_values('time', inplace=True)

# Filter to backtest period
mvrv_sth_daily = mvrv_sth_daily[mvrv_sth_daily['time'] >= START_DATE].copy()
sopr_sth_daily = sopr_sth_daily[sopr_sth_daily['time'] >= START_DATE].copy()
realized_profit_daily = realized_profit_daily[realized_profit_daily['time'] >= START_DATE].copy()
realized_loss_daily = realized_loss_daily[realized_loss_daily['time'] >= START_DATE].copy()
price_daily = price_daily[price_daily['time'] >= START_DATE].copy()
funding_rate_daily = funding_rate_daily[funding_rate_daily['time'] >= START_DATE].copy()
liq_long_daily = liq_long_daily[liq_long_daily['time'] >= START_DATE].copy()
liq_short_daily = liq_short_daily[liq_short_daily['time'] >= START_DATE].copy()

print(f"✅ Loaded {len(price_daily):,} daily bars")
print(f"   Range: {price_daily['time'].min().date()} → {price_daily['time'].max().date()}")

---

## 2. Load Hourly Data (Exits)

Load hourly metrics for faster exit signals.

In [ ]:
# Load hourly metrics for exit signals
print("Loading hourly metrics...")

# On-chain hourly (Bitcoin Lab)
sopr_lth_hourly = pd.read_parquet(BL_HOURLY / 'sopr_lth.parquet')
mvrv_sth_hourly = pd.read_parquet(BL_HOURLY / 'mvrv_sth.parquet')
price_hourly = pd.read_parquet(BL_HOURLY / 'price.parquet')

# Normalize
for df in [sopr_lth_hourly, mvrv_sth_hourly, price_hourly]:
    if 'time' not in df.columns:
        df.reset_index(inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    df.sort_values('time', inplace=True)

# Filter to backtest period
sopr_lth_hourly = sopr_lth_hourly[sopr_lth_hourly['time'] >= START_DATE].copy()
mvrv_sth_hourly = mvrv_sth_hourly[mvrv_sth_hourly['time'] >= START_DATE].copy()
price_hourly = price_hourly[price_hourly['time'] >= START_DATE].copy()

print(f"✅ Loaded {len(price_hourly):,} hourly bars")
print(f"   Range: {price_hourly['time'].min()} → {price_hourly['time'].max()}")

---

## 3. Calculate Buy The Dip Entry Signals (Daily)

Requires **4 out of 5 conditions**:

1. STH-MVRV < 1.0 (underwater)
2. STH-SOPR < 1.0 (selling at loss)
3. Realized P/L Ratio < 1.0 (losses dominate)
4. Funding Rate ≤ 0 (neutral/bearish derivatives)
5. Long Liquidations > Short Liquidations

In [ ]:
# Merge daily data
daily_data = price_daily[['time', 'value']].rename(columns={'value': 'price'})

daily_data = daily_data.merge(
    mvrv_sth_daily[['time', 'value']].rename(columns={'value': 'mvrv_sth'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    sopr_sth_daily[['time', 'value']].rename(columns={'value': 'sopr_sth'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    realized_profit_daily[['time', 'value']].rename(columns={'value': 'realized_profit'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    realized_loss_daily[['time', 'value']].rename(columns={'value': 'realized_loss'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    funding_rate_daily[['time', 'value']].rename(columns={'value': 'funding_rate'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    liq_long_daily[['time', 'value']].rename(columns={'value': 'liq_long'}),
    on='time', how='left'
)
daily_data = daily_data.merge(
    liq_short_daily[['time', 'value']].rename(columns={'value': 'liq_short'}),
    on='time', how='left'
)

# Calculate derived metrics
daily_data['rpl_ratio'] = daily_data['realized_profit'] / daily_data['realized_loss']
daily_data['liq_ratio'] = daily_data['liq_long'] / daily_data['liq_short']

# Calculate Buy The Dip conditions
daily_data['cond1_sth_mvrv'] = daily_data['mvrv_sth'] < 1.0
daily_data['cond2_sth_sopr'] = daily_data['sopr_sth'] < 1.0
daily_data['cond3_rpl_ratio'] = daily_data['rpl_ratio'] < 1.0
daily_data['cond4_funding'] = daily_data['funding_rate'] <= 0.0
daily_data['cond5_liquidations'] = daily_data['liq_ratio'] > 1.0

# Count conditions met
daily_data['conditions_met'] = (
    daily_data['cond1_sth_mvrv'].astype(int) +
    daily_data['cond2_sth_sopr'].astype(int) +
    daily_data['cond3_rpl_ratio'].astype(int) +
    daily_data['cond4_funding'].astype(int) +
    daily_data['cond5_liquidations'].astype(int)
)

# Entry signal: 4+ conditions
daily_data['entry_signal'] = daily_data['conditions_met'] >= 4

print(f"✅ Buy The Dip signals calculated")
print(f"   Total entry signals: {daily_data['entry_signal'].sum()}")
print(f"   Signal frequency: {daily_data['entry_signal'].sum() / len(daily_data) * 100:.1f}%")

# Show sample signals
signals = daily_data[daily_data['entry_signal']][['time', 'price', 'conditions_met']]
print(f"\n📊 Sample entry signals:")
print(signals.head(10))

---

## 4. Define Hourly Exit Strategies

Test multiple hourly exit signals:

### Strategy A: LTH-SOPR Spike
- Exit when hourly LTH-SOPR > 1.5 (profit-taking)

### Strategy B: STH-MVRV Overheated
- Exit when hourly STH-MVRV > 2.0 (euphoria)

### Strategy C: Combined Confirmation
- Exit when LTH-SOPR > 1.3 AND STH-MVRV > 1.5

In [ ]:
# Merge hourly data
hourly_data = price_hourly[['time', 'value']].rename(columns={'value': 'price'})

hourly_data = hourly_data.merge(
    sopr_lth_hourly[['time', 'value']].rename(columns={'value': 'sopr_lth'}),
    on='time', how='left'
)
hourly_data = hourly_data.merge(
    mvrv_sth_hourly[['time', 'value']].rename(columns={'value': 'mvrv_sth'}),
    on='time', how='left'
)

# Calculate exit signals
hourly_data['exit_a_lth_sopr'] = hourly_data['sopr_lth'] > 1.5
hourly_data['exit_b_sth_mvrv'] = hourly_data['mvrv_sth'] > 2.0
hourly_data['exit_c_combined'] = (
    (hourly_data['sopr_lth'] > 1.3) & 
    (hourly_data['mvrv_sth'] > 1.5)
)

print(f"✅ Hourly exit signals calculated")
print(f"\n📊 Exit signal frequency:")
print(f"   Strategy A (LTH-SOPR > 1.5):        {hourly_data['exit_a_lth_sopr'].sum():>6,} hours ({hourly_data['exit_a_lth_sopr'].sum() / len(hourly_data) * 100:>5.1f}%)")
print(f"   Strategy B (STH-MVRV > 2.0):        {hourly_data['exit_b_sth_mvrv'].sum():>6,} hours ({hourly_data['exit_b_sth_mvrv'].sum() / len(hourly_data) * 100:>5.1f}%)")
print(f"   Strategy C (Combined):              {hourly_data['exit_c_combined'].sum():>6,} hours ({hourly_data['exit_c_combined'].sum() / len(hourly_data) * 100:>5.1f}%)")

---

## 5. Multi-Timeframe Backtest Engine

Backtest with:
- **Daily entries**: Buy The Dip (4/5 conditions)
- **Hourly exits**: Check hourly bars for exit signals

In [ ]:
def backtest_multi_timeframe(daily_df, hourly_df, exit_strategy='exit_a_lth_sopr'):
    """
    Backtest with daily entries and hourly exits.
    
    Parameters:
    -----------
    daily_df : DataFrame with daily entry signals
    hourly_df : DataFrame with hourly exit signals
    exit_strategy : Column name for exit signal in hourly_df
    
    Returns:
    --------
    trades : List of trade dictionaries
    equity_curve : DataFrame with portfolio value over time
    """
    
    trades = []
    position = None
    initial_capital = 10_000
    capital = initial_capital
    
    # Create equity curve on daily basis
    equity_curve = daily_df[['time', 'price']].copy()
    equity_curve['capital'] = initial_capital
    equity_curve['position'] = 0.0
    
    for idx, day in daily_df.iterrows():
        date = day['time']
        price = day['price']
        
        # Entry logic: Check for new entry signal
        if position is None and day['entry_signal']:
            # Enter position
            position = {
                'entry_date': date,
                'entry_price': price,
                'entry_capital': capital,
                'size': capital / price  # Full allocation
            }
            
        # Exit logic: Check hourly bars for exit signal
        if position is not None:
            # Get hourly bars for this day
            day_start = pd.Timestamp(date).normalize()
            day_end = day_start + pd.Timedelta(days=1)
            
            hourly_day = hourly_df[
                (hourly_df['time'] >= day_start) & 
                (hourly_df['time'] < day_end)
            ]
            
            # Check if any hourly bar triggers exit
            exit_triggered = hourly_day[exit_strategy].any() if len(hourly_day) > 0 else False
            
            if exit_triggered:
                # Exit on first hourly signal of the day
                exit_hour = hourly_day[hourly_day[exit_strategy]].iloc[0]
                exit_price = exit_hour['price']
                exit_time = exit_hour['time']
                
                # Calculate trade P&L
                exit_value = position['size'] * exit_price
                pnl = exit_value - position['entry_capital']
                pnl_pct = (exit_price / position['entry_price'] - 1) * 100
                hold_days = (exit_time - position['entry_date']).days
                
                # Record trade
                trades.append({
                    'entry_date': position['entry_date'],
                    'entry_price': position['entry_price'],
                    'exit_date': exit_time,
                    'exit_price': exit_price,
                    'hold_days': hold_days,
                    'pnl_pct': pnl_pct,
                    'pnl_usd': pnl
                })
                
                # Update capital
                capital = exit_value
                position = None
        
        # Update equity curve
        if position is not None:
            # Mark-to-market
            current_value = position['size'] * price
            equity_curve.loc[equity_curve['time'] == date, 'capital'] = current_value
            equity_curve.loc[equity_curve['time'] == date, 'position'] = 1
        else:
            equity_curve.loc[equity_curve['time'] == date, 'capital'] = capital
            equity_curve.loc[equity_curve['time'] == date, 'position'] = 0
    
    return trades, equity_curve

print("✅ Backtest engine defined")

---

## 6. Run Backtests

Test all exit strategies and compare performance.

In [ ]:
# Run backtests for all strategies
strategies = [
    ('Strategy A: LTH-SOPR > 1.5', 'exit_a_lth_sopr'),
    ('Strategy B: STH-MVRV > 2.0', 'exit_b_sth_mvrv'),
    ('Strategy C: Combined', 'exit_c_combined'),
]

results = {}

print("Running backtests...\n")
print("=" * 80)

for name, exit_col in strategies:
    trades, equity_curve = backtest_multi_timeframe(
        daily_data, 
        hourly_data, 
        exit_strategy=exit_col
    )
    
    results[name] = {
        'trades': trades,
        'equity_curve': equity_curve,
        'exit_strategy': exit_col
    }
    
    # Calculate metrics
    if len(trades) > 0:
        trades_df = pd.DataFrame(trades)
        
        total_return = (equity_curve['capital'].iloc[-1] / 10_000 - 1) * 100
        num_trades = len(trades)
        win_rate = (trades_df['pnl_pct'] > 0).sum() / len(trades_df) * 100
        avg_return = trades_df['pnl_pct'].mean()
        avg_hold = trades_df['hold_days'].mean()
        
        print(f"{name}")
        print(f"  Total Return:   {total_return:>8.1f}%")
        print(f"  Trades:         {num_trades:>8}")
        print(f"  Win Rate:       {win_rate:>8.1f}%")
        print(f"  Avg Return:     {avg_return:>8.1f}%")
        print(f"  Avg Hold:       {avg_hold:>8.1f} days")
        print("-" * 80)
    else:
        print(f"{name}")
        print("  No trades executed")
        print("-" * 80)

print("\n✅ All backtests complete")

---

## 7. Performance Comparison

Compare vs baselines:
1. Buy & Hold
2. Daily entries + Daily exits

In [ ]:
# Buy & Hold baseline
bh_return = (daily_data['price'].iloc[-1] / daily_data['price'].iloc[0] - 1) * 100

print("="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(f"\n📊 Buy & Hold:          {bh_return:>8.1f}%")
print("-"*80)

for name, data in results.items():
    if len(data['trades']) > 0:
        total_return = (data['equity_curve']['capital'].iloc[-1] / 10_000 - 1) * 100
        excess_return = total_return - bh_return
        print(f"{name}")
        print(f"  Total Return:   {total_return:>8.1f}%")
        print(f"  vs Buy & Hold:  {excess_return:>+8.1f}%")
        print("-"*80)

print("\n✅ Comparison complete")

---

## 8. Visualizations

In [ ]:
# Plot equity curves
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Equity curves
ax1 = axes[0]
for name, data in results.items():
    equity = data['equity_curve']
    ax1.plot(equity['time'], equity['capital'], label=name, linewidth=2)

# Buy & Hold
bh_equity = daily_data['price'] / daily_data['price'].iloc[0] * 10_000
ax1.plot(daily_data['time'], bh_equity, label='Buy & Hold', 
         linewidth=2, linestyle='--', alpha=0.7, color='gray')

ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.set_title('Equity Curves: Daily Entries + Hourly Exits', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Price with entry/exit markers (Strategy A)
ax2 = axes[1]
ax2.plot(daily_data['time'], daily_data['price'], color='black', linewidth=1.5, label='BTC Price')

# Plot trades for Strategy A
if 'Strategy A: LTH-SOPR > 1.5' in results and len(results['Strategy A: LTH-SOPR > 1.5']['trades']) > 0:
    trades_a = pd.DataFrame(results['Strategy A: LTH-SOPR > 1.5']['trades'])
    
    # Entry markers
    ax2.scatter(trades_a['entry_date'], trades_a['entry_price'], 
               color='green', marker='^', s=100, label='Entry', zorder=5)
    
    # Exit markers
    ax2.scatter(trades_a['exit_date'], trades_a['exit_price'],
               color='red', marker='v', s=100, label='Exit', zorder=5)

ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('BTC Price ($)', fontsize=12)
ax2.set_title('Strategy A: Entry/Exit Points', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("✅ Charts complete")

---

## 9. Trade Analysis

In [ ]:
# Detailed trade analysis for best strategy
best_strategy = max(results.items(), 
                   key=lambda x: x[1]['equity_curve']['capital'].iloc[-1] if len(x[1]['trades']) > 0 else 0)

print("="*80)
print(f"BEST STRATEGY: {best_strategy[0]}")
print("="*80)

if len(best_strategy[1]['trades']) > 0:
    trades_df = pd.DataFrame(best_strategy[1]['trades'])
    
    # Summary statistics
    print(f"\n📊 Trade Statistics:")
    print(f"  Total Trades:      {len(trades_df)}")
    print(f"  Winners:           {(trades_df['pnl_pct'] > 0).sum()}")
    print(f"  Losers:            {(trades_df['pnl_pct'] <= 0).sum()}")
    print(f"  Win Rate:          {(trades_df['pnl_pct'] > 0).sum() / len(trades_df) * 100:.1f}%")
    print(f"\n  Average Return:    {trades_df['pnl_pct'].mean():.1f}%")
    print(f"  Median Return:     {trades_df['pnl_pct'].median():.1f}%")
    print(f"  Best Trade:        {trades_df['pnl_pct'].max():.1f}%")
    print(f"  Worst Trade:       {trades_df['pnl_pct'].min():.1f}%")
    print(f"\n  Avg Hold Period:   {trades_df['hold_days'].mean():.1f} days")
    print(f"  Median Hold:       {trades_df['hold_days'].median():.1f} days")
    print(f"  Max Hold:          {trades_df['hold_days'].max():.0f} days")
    print(f"  Min Hold:          {trades_df['hold_days'].min():.0f} days")
    
    # Show all trades
    print(f"\n📋 All Trades:")
    print("-"*80)
    display_df = trades_df.copy()
    display_df['entry_date'] = display_df['entry_date'].dt.strftime('%Y-%m-%d')
    display_df['exit_date'] = display_df['exit_date'].dt.strftime('%Y-%m-%d %H:%M')
    print(display_df.to_string(index=False))
else:
    print("\nNo trades executed")

print("\n" + "="*80)

---

## 10. Conclusions & Next Steps

### Key Findings

**[To be filled after running backtest]**

1. Did hourly exits improve performance vs daily exits?
2. Which hourly exit signal worked best?
3. What was the impact on:
   - Total return
   - Max drawdown
   - Win rate
   - Average hold period

### Next Steps

1. **Test more exit strategies**:
   - Price momentum breaks (hourly MA crossovers)
   - Realized profit spikes (hourly)
   - Combined on-chain + derivatives signals

2. **Optimize thresholds**:
   - LTH-SOPR exit level (1.3, 1.5, 1.8)
   - STH-MVRV exit level (1.5, 2.0, 2.5)
   - Confirmation requirements

3. **Risk management**:
   - Add stop losses
   - Position sizing based on signal strength
   - Trailing stops on hourly data

4. **Paper trading**:
   - Implement live monitoring
   - Test with real-time hourly data
   - Alert system for signals

---